In [ ]:
################################################
# New style 
################################################
import sys
import os

workdir_ = '/glade/work/juliob'
if ( workdir_ not in sys.path ):
    sys.path.append(workdir_)
    print( f" a path to {workdir_} added in {__name__} ")

from myPythonTools.Utils import utils as uti
from myPythonTools.Utils import numerical_utils as nuti
from myPythonTools.Utils import AveragingUtils as Av
from myPythonTools.Utils import validation_data as Val
from myPythonTools.Utils import PlotUtil as Pu
from myPythonTools.Plotting import LatLonMaps as LL 
from myPythonTools.Plotting import utils as puti
from myPythonTools.CASutils import filter_utils as fu 

from PyRegridding.Utils import GridUtils as GrU
from PyRegridding.Utils import MakePressures as MkP
from PyRegridding.Drivers import RegridField as RgF


# The usual
from datetime import date
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.tri as tri

# Cartopy for pretty maps
import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Some other useful packages 
import importlib
import copy
import time
import cftime

importlib.reload( uti )
importlib.reload( puti )
importlib.reload( nuti )
importlib.reload( Pu )
importlib.reload(Av)
importlib.reload(Val)
#importlib.reload(vAB)
importlib.reload(MkP)
importlib.reload(RgF)


In [ ]:
from myPythonTools.YAML import get_exp_list as Xl
importlib.reload(Xl)


In [ ]:
print( sys.path )
############################
# Initialize dict for regridding
############################
RgObs={}


In [ ]:
### Make a list of files
import yaml
#from box import Box


with open('../YAML/configure_plotting.yaml', 'r') as file:
    cfgPlot = yaml.safe_load(file)


######################################################
season =  'jja' #cfgPlot['zonal_mean']['season']
valData  = cfgPlot['zonal_mean']['validation']['data']
valYears = cfgPlot['zonal_mean']['validation']['years']
print( f"Season= {season}" )
print( valData , valYears )
fld='U' #'U'
hsPat = 'cam.h0a' # assume fld is in same h-file for all exps. Not great.


regrid_val_x_model = False # True
all_x_fv1x1_zERA = False #True #True
use_desc_in_title = False #True
use_pmid_for_z =False #False
plot_jet_axis_range = False #True
use_ymdPat_from_yaml = True

In [ ]:


which_figure = 4
if (which_figure==1):
    exp_file,xl= Xl.Fig1Paper() 
elif (which_figure==4):
    exp_file,xl= Xl.KeyFig_Paper() 
elif (which_figure==99):
    exp_file,xl= Xl.ControlRuns() 
else:
    print( 'no valid figure')

print( xl )

exp_file=f'../YAML/{exp_file}'
with open( exp_file, 'r') as file:
    cfg = yaml.safe_load(file)

Al =[]
for x in xl:
    exp, subd, Src, Hkey, Dst, useri = cfg[x]['name'] , cfg[x]['subdir'] , cfg[x]['SrcGrid'] , cfg[x]['Hkey'] , cfg[x]['DstGrid'] , cfg[x]['user'] 

    if (use_ymdPat_from_yaml==True):
        ymdPat=cfg[x]['ymdPat']
        if ( 'ymdPat_is_range' in cfg[x] ):
            ymdPat_is_range = cfg[x]['ymdPat_is_range']
        else:
            ymdPat_is_range = False
    else:
        ymdPat = '*' # ['1985-*', '1986-*']
        ymdPat_is_range = False

    
    
    print( exp, subd, Src, Hkey, Dst, useri )
    print( ymdPat  )
    
    A = uti.MakeDict4Exp( exp=exp , user=useri, subd=subd , 
                        hsPat=hsPat , ymdPat=ymdPat , 
                        ymdPat_is_range =ymdPat_is_range , 
                        cmip_fld=fld ,
                        verbose=True , open_dataset=True )


    
    A['alias']=x
    A['Src']=Src
    if ('SrcVgrid' in cfg[x]):
        A['SrcVgrid']=cfg[x]['SrcVgrid']
        print( f"SrcVgrid for {x} = {A['SrcVgrid']} ")
    else:
        print( f'No SrcVgrid in {x}' )
    A['Hkey']=Hkey
    if 'gw_parameters' in cfg[x]:
        A['gw_parameters'] = cfg[x]['gw_parameters']
    if 'desc' in cfg[x]:
        A['desc'] = cfg[x]['desc']
    Al.append( A )
    print( f'Vertcal levs - {A.X.lev.shape}' )
    print( f'{A.exp} Years {len(A.X.time) /12. } \n')


In [ ]:
for A in Al:
    print( 'PINT' in list(A.X))

In [ ]:
months,years = Av.MonthsInDataset( Al[0].X )
print( months, years )

In [ ]:
#print(np.shape(Al[1]['PMID_s']))
from myPythonTools.Zany import jetaxis as jax
importlib.reload( jax )

for A in Al:
    AA_tzy,lat_a,lon_a = jax.make_seasonal_zonal_tzyCube( X=A.X , fld=fld, Src=A.Src , Dst='fv1x1' )
    AA_szy  = np.std(AA_tzy , axis=0)
    AA_mzy = np.mean(AA_tzy , axis=0)

    lev_a=A.X.lev.values
    zlev_a=-7.0*np.log( lev_a/1_000. )

    A['AA_mzy']=AA_mzy
    A['AA_szy']=AA_szy
    A['lat']=lat_a
    A['lon']=lon_a
    A['lev']=lev_a
    A['zlev']=zlev_a



In [ ]:
%%time
Xval =xr.open_mfdataset( '/glade/campaign/cgd/amp/juliob/ERA5/fv1x1_L93_monthly/ZYX/ERA5.fv1x1_L93.time*.nc' )

In [ ]:
%%time
AvalA_tzy,lat_a,lon_a = jax.make_seasonal_zonal_tzyCube( X=Xval , fld='U', Src='fv1x1' , Dst='fv1x1' )

In [ ]:

AvalA_szy  = np.std(AvalA_tzy , axis=0)
AvalA_mzy = np.mean(AvalA_tzy , axis=0)

Aval={ 'AA_mzy':AvalA_mzy }
Aval['AA_szy'] = AvalA_szy

In [ ]:
fig, ax = plt.subplots(1,3,figsize=(20,8))
ax[0].contour( Al[0]['lat'], Al[0]['zlev'] , Al[0]['AA_mzy'] )
ax[1].contour( Al[1]['lat'], Al[1]['zlev'] , Al[1]['AA_mzy'] )
ax[2].contour( Al[2]['lat'], Al[2]['zlev'] , Al[2]['AA_mzy'] )
#ax[1].contour( lat_a, zlev_a , AvalA_mzy )
#ax[2].contour( lat_a, zlev_a , mean-AvalA_mzy )


In [ ]:



# Get the default color cycle
#colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
#colors = plt.cm.tab10(np.arange(10))  # RGBA array
colors = ['C0','C1','C2','C3','C4','C5'] # RGBA array

fig, ax = plt.subplots()
z=10
# Plot control (blue)

for i in np.arange( len(Al) ):
    A=Al[i]
    print(i)
    mean, std, lat_a, zlev_a = A['AA_mzy'],A['AA_szy'],A['lat'],A['zlev']
    z = np.abs(zlev_a - 43.0).argmin()
    print( z, zlev_a[z] )
    ax.plot(lat_a, mean[z,:], color=colors[i], label="control")
    ax.fill_between(lat_a, mean[z,:] - 2*std[z,:], mean[z,:] + 2*std[z,:],
                    color=colors[i], alpha=0.3, label="±2 Std Dev")

i+1
meanv, stdv = Aval['AA_mzy'],Aval['AA_szy']
ax.plot(lat_a, meanv[z,:], color=colors[i] , label="control")
ax.fill_between(lat_a, meanv[z,:] - 2*stdv[z,:], meanv[z,:] + 2*stdv[z,:],
                color=colors[i], alpha=0.3, label="±2 Std Dev")

plt.legend()

In [ ]:
Al[0].desc='CAM7 w/ new GW source'
Al[1].desc='CAM7 default GW sources'


In [ ]:



# Get the default color cycle
#colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
#colors = plt.cm.tab10(np.arange(10))  # RGBA array
colors = ['C0','C1','C2','C3','C4','C5'] # RGBA array

fig, ax = plt.subplots(1,3, figsize=(21,7))

# Plot control (blue)
unit = "m s$^{-1}$"

for i in np.arange( len(Al) ):
    A=Al[i]
    print(i)
    mean, std, lat_a, zlev_a = A['AA_mzy'],A['AA_szy'],A['lat'],A['zlev']
    z = np.abs(zlev_a - 40.0).argmin()
    print( z, zlev_a[z] )
    ax[i].plot(lat_a, mean[z,:],  label=A.desc)
    ax[i].fill_between(lat_a, mean[z,:] - 2*std[z,:], mean[z,:] + 2*std[z,:],
                     alpha=0.3, label="±2 Std Dev")

    
    z = np.abs(Al[0].zlev - 40.0).argmin()
    meanv, stdv = Aval['AA_mzy'],Aval['AA_szy']
    ax[i].plot(lat_a, meanv[z,:],  label="ERA5")
    ax[i].fill_between(lat_a, meanv[z,:] - 2*stdv[z,:], meanv[z,:] + 2*stdv[z,:],
                     alpha=0.3, label="±2 Std Dev")
    ax[i].set_xlim(-90,0)
    ax[i].set_ylim(-20,120)
    if i==0:
        ax[i].set_ylabel(f"U(40km) ({unit})",fontsize=18 )
    ax[i].set_xlabel(f"Latitude",fontsize=18 )
    #plt.ylabel("Z (km)",fontsize=18 )
    ax[i].tick_params(axis='both', which='major', labelsize=18)
    
    ax[i].legend()

FigName = f"Errors_at_40km_{fld}.png"

print( FigName) 
Pu.save_link(png_filename = FigName, dpi=600 )

In [ ]:
#### STOP HERE FOR A MOMENT

In [ ]:
#### STOP HERE FOR A MOMENT

In [ ]:
AlCopy=copy.deepcopy(Al)

In [ ]:
#########
#   T H E   E N D
#########